# Bear Stories with GRPO

Post-trains `TinyStories-33M` with Group Relative Policy Optimization (GRPO) so it writes stories about bears. Each section checks one component with a small printout or plot; Section 8 runs the $2\times 2$ reward × clipping experiment.

## 0. Setup

**Colab:** put this folder in Google Drive as `MyDrive/bear-stories-with-grpo`, open this notebook, select a **T4 GPU** runtime, then **Run all** and approve Drive access. Code, downloaded models, and results live in the Drive folder.

**Locally:** start Jupyter from this folder; the first cell then only starts the timer.

The final cell saves runtime measurements; total time includes setup.

In [ ]:
# Google Colab: mount Drive and open the project folder. Locally this cell only starts the timer.
import time
NOTEBOOK_STARTED = time.perf_counter()

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    import os
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/bear-stories-with-grpo")


In [ ]:
import sys
import importlib
from pathlib import Path

assert Path("setup.py").exists(), "Change into the project folder before continuing."
%pip install -q -e ".[semantic]"

# Compatibility fix for Colab's older IPython on Python 3.12+
sys.modules.setdefault("imp", importlib)

%load_ext autoreload
%autoreload 2

RESULTS_ROOT = Path("runs")
RESULTS_ROOT.mkdir(exist_ok=True)


In [ ]:
import torch
import matplotlib.pyplot as plt

from train import (
    ABLATION_PREFIXES,
    EVAL_PREFIXES,
    TRAIN_PREFIXES,
    TrainConfig,
    load_policy,
)
from utils import (
    DEFAULT_ARMS,
    discover_latest_runs,
    ensure_tinystories_checkpoint,
    plot_training_curves,
    print_run_timings,
    print_table1_bear_rates,
    run_arm,
    run_grid,
    show_saved_examples,
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)
print("TRAIN_PREFIXES:", TRAIN_PREFIXES)
print("EVAL_PREFIXES:", EVAL_PREFIXES)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Overview

The language model is the policy: the prompt plus the text generated so far is the state, and each generated token is an action. GRPO samples a group of completions per prompt, scores each with a reward, and makes the above-average completions more likely.

**Files**

- `bear_stories_grpo.ipynb` — this workflow (checks, experiments, plots).
- `grpo.py` — completion log-probs, group-relative advantages, clipped GRPO loss, k3 KL penalty.
- `rewards.py` — sparse (keyword) and dense (sentence-embedding) rewards.
- `train.py` — grouped rollouts, the GRPO optimizer step, and the training loop.
- `eval.py` — evaluation metrics.
- `utils.py` — experiment and plotting helpers.
- `tests.py` — unit tests.
- `setup.py` — package install.
- `configs/grid_2x2.json` — 2×2 experiment hyperparameters.

**Openers (fixed in `train.py`)**

- Train (5): `Once upon a time`, `Deep in the forest`, `One sunny morning`, `In a small village`, `Late one afternoon`.
- Eval (3): `Once upon a time`, `Deep in the forest`, held-out `Late one night`.

## 2. TinyStories and Language Models as Policies

Load the pretrained policy and inspect a few completions before training.

In [ ]:
from eval import evaluate

local_dir = ensure_tinystories_checkpoint()
model, tokenizer = load_policy(local_dir, device=device)
model.eval()

baseline_preview = evaluate(
    model,
    tokenizer,
    list(EVAL_PREFIXES),
    count=1,
    max_new_tokens=80,
    seed=0,
    compute_metrics=False,
)
for index, sample in enumerate(baseline_preview["samples"], start=1):
    print(f"\n--- baseline story {index} [{sample['prompt']}] ---")
    print(sample["completion"])

### Sequence log probabilities

This cell prints the log probability `sequence_log_probs` selects for every completion token.

In [ ]:
from grpo import sequence_log_probs

prompt_text = "Once upon a time"
completion_text = ", there was a bear."
prompt_batch = tokenizer(prompt_text, return_tensors="pt")
target_ids = tokenizer(
    completion_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids
valid_tokens = torch.ones_like(target_ids, dtype=torch.bool)

with torch.no_grad():
    selected_log_probs = sequence_log_probs(
        model,
        prompt_batch.input_ids,
        prompt_batch.attention_mask,
        target_ids,
        valid_tokens,
    )

for token_id, log_prob in zip(target_ids[0], selected_log_probs[0]):
    token = tokenizer.decode([int(token_id)])
    print(f"{token!r:14s} log p = {float(log_prob):7.3f}   p = {float(log_prob.exp()):.4f}")

## 3. Reward Functions

Compute rewards from completions only. The next two steps inspect the two reward definitions separately.

### Sparse reward: does the word "bear" exist?

Below are six handwritten completions. Pretend they are model outputs and check that only whole-word occurrences of `bear` or `bears` receive reward.

In [ ]:
from rewards import SparseBearReward

example_completions = [
    "A bear helped Lily find her way home.",
    "Lily saw a furry animal beside the river.",
    "Sam saw a cute cub running around.",
    "Tom played with a bright red ball.",
    "bear bear bear bear bear",
    "That statue resemble an ursine animal",  # ursine means bear-like
]
sparse_scores = SparseBearReward()(example_completions)
for text, score in zip(example_completions, sparse_scores):
    print(f"sparse={float(score):.1f} | {text}")

### Dense reward: how similar is the completion to the word "bear"?

Compare the dense scores across the same completions, then compare how the two rewards rank them. Their raw scales are different.

In [ ]:
from rewards import DenseStoryReward

dense_scores = DenseStoryReward(device=device)(example_completions)
x = torch.arange(len(example_completions))
plt.figure(figsize=(8, 3.5))
plt.bar(x - 0.18, sparse_scores, width=0.36, label="sparse")
plt.bar(x + 0.18, dense_scores, width=0.36, label="dense")
plt.xticks(x, [f"story {i + 1}" for i in x])
plt.ylabel("reward")
plt.legend()
plt.show()

for text, score in zip(example_completions, dense_scores):
    print(f"dense={float(score):.3f} | {text}")

### Reward the base-model completions

Generate a few stories from the pretrained model and score them with both rewards.

In [ ]:
baseline_preview = evaluate(
    model,
    tokenizer,
    list(EVAL_PREFIXES),
    count=1,
    max_new_tokens=256,
    seed=0,
    compute_metrics=True,
)
arr_baseline_preview = [sample["completion"] for sample in baseline_preview["samples"]]
sparse_scores = SparseBearReward()(arr_baseline_preview)
dense_scores = DenseStoryReward(device=device)(arr_baseline_preview)

for index, sample in enumerate(baseline_preview["samples"], start=1):
    print(f"\n--- baseline story {index} [{sample['prompt']}] ---")
    print(sample["completion"])
    print(
        "Rewards:",
        f"sparse={float(sparse_scores[index - 1]):.1f},",
        f"dense={float(dense_scores[index - 1]):.3f}",
    )

## 4. Group Relative Policy Optimization

Each GRPO component, checked on small inputs.

### Group-relative advantages

A **group** contains the $G$ completions sampled from the same prompt. This example uses two prompts with $G=4$: entries 0–3 have `group_id=0`, and entries 4–7 have `group_id=1`. GRPO normalizes rewards separately inside each group.

Change `group_rewards` to try different scores. As an edge-case experiment, make all four rewards in one group equal and observe that every advantage in that group becomes zero.

In [ ]:
from grpo import group_advantages

group_rewards = torch.tensor([
    0.0, 0.2, 0.8, 1.0,  # group 0
    0.1, 0.4, 0.4, 0.9,  # group 1
])
group_ids = torch.tensor([
    0, 0, 0, 0,
    1, 1, 1, 1,
])
advantages = group_advantages(group_rewards, group_ids)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
group_colors = ["tab:blue", "tab:orange"]
for group, ax in enumerate(axes):
    in_group = group_ids == group
    group_values = advantages[in_group]
    members = torch.arange(1, len(group_values) + 1)
    ax.bar(members, group_values, color=group_colors[group], alpha=0.75)
    ax.scatter(members, group_values, color=group_colors[group], s=45, zorder=3)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"prompt group {group}")
    ax.set_xlabel("completion within group")
    ax.set_xticks(members)
    ax.set_ylim(-2, 2)
axes[0].set_ylabel("advantage")
fig.tight_layout()
plt.show()

for group in range(2):
    in_group = group_ids == group
    print(f"group {group} rewards:   ", group_rewards[in_group].tolist())
    print(f"group {group} advantages:", advantages[in_group].tolist())

### Importance weighting and clipping

The solid curves show the clipped surrogate. The dashed curves show the unclipped surrogate.

In [ ]:
from grpo import grpo_loss

ratios = torch.linspace(0.5, 1.5, 41)
mask = torch.ones(1, 1, dtype=torch.bool)

def surrogate_values(advantage, clip_epsilon):
    values = []
    for ratio in ratios:
        loss, _ = grpo_loss(
            ratio.log().reshape(1, 1),
            torch.zeros(1, 1),
            mask,
            torch.tensor([advantage]),
            clip_epsilon=clip_epsilon,
        )
        values.append(-float(loss))
    return values

plt.figure(figsize=(7, 3.5))
for advantage in (1.0, -1.0):
    plt.plot(ratios, surrogate_values(advantage, None), "--", label=f"A={advantage:+.0f}, no clip")
    plt.plot(ratios, surrogate_values(advantage, 0.2), label=f"A={advantage:+.0f}, clip")
plt.axvline(0.8, color="gray", linewidth=1)
plt.axvline(1.2, color="gray", linewidth=1)
plt.xlabel("importance weight $w$")
plt.ylabel("surrogate")
plt.title("GRPO clipping")
plt.legend(fontsize=8)
plt.show()

### KL regularization

The sampled KL penalty should be nonnegative and reach zero when the policy and reference log probabilities agree. The main $2\times 2$ experiment uses $\beta=0$; this cell checks `kl_penalty` on its own.

In [ ]:
from grpo import kl_penalty

log_differences = torch.linspace(-1.0, 1.0, 41)
kl_values = [
    float(kl_penalty(torch.tensor([[-difference]]), torch.zeros(1, 1), mask))
    for difference in log_differences
]

plt.figure(figsize=(7, 3.5))
plt.plot(log_differences, kl_values)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("log p(reference) - log p(policy)")
plt.ylabel("penalty")
plt.title("Sampled KL penalty")
plt.show()

## 5. Training Loop

Each update in `train_grpo` (`train.py`) samples a group of completions per prompt, scores them, computes group-relative advantages, and runs `grpo_step` for several optimizer epochs on that fixed rollout. The smoke test later in the notebook is optional.

## 6. Evaluation

The metrics used to compare the pretrained and post-trained models.

### Completion metrics

Use this fixed set of completion records to check the completion-only metrics.

In [ ]:
from eval import completion_metrics

metric_examples = [
    {"bear_mentions": 1, "reference_nll": 2.0, "completion_token_count": 4},
    {"bear_mentions": 0, "reference_nll": 3.0, "completion_token_count": 6},
]
print(completion_metrics(metric_examples))

### Aggregate evaluation

This small evaluation calls the aggregate metric code on generated stories without training a model.

In [ ]:
evaluation_preview = evaluate(
    model,
    tokenizer,
    list(EVAL_PREFIXES),
    count=2,
    max_new_tokens=40,
    seed=1,
)
print(evaluation_preview["aggregate"])

In [ ]:
smoke_config = TrainConfig(
    updates=2,
    group_size=4,
    inner_epochs=1,
    max_new_tokens=32,
    seed=17,
)
print(smoke_config)

## 7. Testing and smoke test

Run the unit tests in `tests.py`. The smoke test is optional: leave `RUN_SMOKE = False` so **Run All** goes straight to the $2\times 2$ experiment after the tests.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "tests.py", "-v"], check=True)


In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    smoke_runs = run_grid(
        arms=["sparse_clip"],
        updates=2,
        eval_count=2,
        output_dir=str(RESULTS_ROOT / "smoke"),
        device=device,
        seed=17,
        max_new_tokens=32,
        group_size=4,
        inner_epochs=1,
    )
    print_run_timings(smoke_runs)
    plot_training_curves(smoke_runs, metric="bear_story_rate", title="Smoke: bear-story rate")
else:
    print("Smoke skipped (RUN_SMOKE=False).")

## 8. Experiments: reward × clipping

The reference configuration uses $G=16$, four optimizer epochs per rollout, 20 GRPO updates, $\beta_{\mathrm{KL}}=0$, and seed **17**.

Set `RUN_2X2 = True` once. After the four arms finish, set `LOAD_EXISTING_2X2 = True` and `RUN_2X2 = False` to replot without retraining.

In [ ]:
RUN_2X2 = True
LOAD_EXISTING_2X2 = False
EXPERIMENT_SEED = 17
OUTPUT_DIR = str(RESULTS_ROOT / f"2x2_seed{EXPERIMENT_SEED}")
ARMS = list(DEFAULT_ARMS)  # sparse_clip, sparse_noclip, dense_clip, dense_noclip

import gc
if "model" in globals():
    del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
EXPERIMENT_STARTED = time.perf_counter()
grid_runs = {}
if RUN_2X2:
    grid_runs = run_grid(
        arms=ARMS,
        updates=20,
        eval_count=25,
        output_dir=OUTPUT_DIR,
        device=device,
        seed=EXPERIMENT_SEED,
    )
elif LOAD_EXISTING_2X2:
    grid_runs = discover_latest_runs(OUTPUT_DIR, arms=ARMS)
    print("Loaded:", grid_runs)
else:
    print("2×2 skipped. Set RUN_2X2 or LOAD_EXISTING_2X2.")

if grid_runs:
    print_run_timings(grid_runs)
EXPERIMENT_SECONDS = time.perf_counter() - EXPERIMENT_STARTED


### Figures 1–3 and Table 1

- **Figure 1:** mean reward vs GRPO update
- **Figure 2:** bear-story rate vs GRPO update
- **Figure 3:** clipping fraction vs GRPO update (constant 0 when clipping is disabled)
- **Table 1:** baseline + four post-training checkpoints × three eval openers + mean

In [ ]:
if not grid_runs:
    raise RuntimeError("No grid_runs in memory. Run the 2×2 cell first (train or load).")

plot_training_curves(grid_runs, metric="reward_mean", title="Figure 1: mean reward")
plot_training_curves(grid_runs, metric="bear_story_rate", title="Figure 2: bear-story rate")
plot_training_curves(grid_runs, metric="clip_fraction", title="Figure 3: clipping fraction")

print("\nTable 1 (bear-story rate)")
print_table1_bear_rates(grid_runs, eval_prompts=EVAL_PREFIXES)

### Read the stories

Print a few post-training completions from each arm (successes and failures). Stories are loaded from `evaluation_samples.jsonl` — no extra generation.

In [ ]:
if not grid_runs:
    raise RuntimeError("No grid_runs in memory. Run the 2×2 cell first (train or load).")

for arm, run_dir in grid_runs.items():
    print(f"\n========== {arm} ==========")
    show_saved_examples(run_dir, n_good=1, n_bad=1)

### Optional: KL penalty sweep

Retrains the dense + clipping arm with $\beta \in \{0, 0.02, 0.1\}$ to trade reward against staying close to the pretrained model. Leave `RUN_KL = False` to skip it.

In [ ]:
RUN_KL = False
KL_ARM = "dense_clip"
KL_BETAS = [0.0, 0.02, 0.1]

kl_runs = {}
if RUN_KL:
    for beta in KL_BETAS:
        label = f"{KL_ARM}_beta_{beta:g}"
        kl_runs[label] = run_arm(
            KL_ARM,
            updates=20,
            eval_count=25,
            output_dir=str(RESULTS_ROOT / "kl_extension"),
            device=device,
            seed=EXPERIMENT_SEED,
            run_name=label,
            kl_beta=beta,
        )
    print_run_timings(kl_runs)
    plot_training_curves(kl_runs, metric="bear_story_rate", title="Optional KL: bear-story rate")
else:
    print("Optional KL skipped (RUN_KL=False).")

## Runtime summary


In [ ]:
import json
import platform
import importlib.metadata
from datetime import datetime, timezone

runtime_report = {
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_seconds_including_setup": time.perf_counter() - NOTEBOOK_STARTED,
    "experiment_seconds": EXPERIMENT_SECONDS,
    "device": device,
    "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ["torch", "transformers", "sentence-transformers", "huggingface_hub", "numpy"]},
    "runs": {arm: str(path) for arm, path in grid_runs.items()},
}
report_path = RESULTS_ROOT / ("runtime_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".json")
report_path.write_text(json.dumps(runtime_report, indent=2) + "\n")
print(json.dumps(runtime_report, indent=2))
print("Saved runtime report:", report_path)
